In [7]:
from astropy.io import ascii
from astropy.table import Table, join
import numpy as np

In [31]:
meta = Table(ascii.read('cube_metadata.csv'))
clusterz = Table(ascii.read('clusters.csv'))
cube_ra = Table(ascii.read('cube_info.csv'))['CUBEKEY','cluster_ra','cluster_dec']

master = join(meta,clusterz)
master = join(master,cube_ra)
master['z'][master['z']==0] = np.nan
for m in master:
    m['TIME'] = m['TIME'][:8]

In [37]:
def round_sig_figs(table, sig_figs):
    """
    Round specified columns to a given number of significant figures.

    Parameters
    ----------
    table : astropy.table.Table
        Modified in place.
    sig_figs : dict
        Mapping of column name to number of sig figs, e.g.
        {"flux": 3, "flux_err": 2, "redshift": 4}
    """
    t = table.copy()
    for col, sf in sig_figs.items():
        if col not in t.colnames:
            print(f"Warning: column '{col}' not found, skipping.")
            continue
        vals = np.array(t[col], dtype=float)
        # sig fig rounding: round to sf s.f. via log10 scaling
        with np.errstate(divide="ignore", invalid="ignore"):
            magnitudes = np.floor(np.log10(np.abs(vals)))
            magnitudes = np.where(np.isfinite(magnitudes), magnitudes, 0)
            decimals = (sf - 1 - magnitudes).astype(int)
        rounded = np.array([
            round(v, int(d)) if np.isfinite(v) else v
            for v, d in zip(vals, decimals)
        ])
        t[col] = rounded
    return t

In [38]:
rounder = {'z':4,'cluster_ra':6,'cluster_dec':6}
preprocessed = round_sig_figs(master, rounder)

In [42]:
preprocessed = preprocessed['OBJECT','DATE','TIME','FWHM_XY_AVG','FWHM_MAX','FWHM_MIN','z','cluster_ra','cluster_dec']
preprocessed

OBJECT,DATE,TIME,FWHM_XY_AVG,FWHM_MAX,FWHM_MIN,z,cluster_ra,cluster_dec
str17,str10,str15,str18,str5,str5,float64,float64,float64
Abell141,2023-08-21,20:53:46,N/A,N/A,N/A,0.23,16.3868,-24.6465
A209,2019-11-25,12:41:47,0.903,1.146,0.781,0.2048,22.9694,-13.6121
Abell2697,2023-09-19,15:47:51,N/A,N/A,N/A,0.234,0.798569,-6.09154
Abell2811,2023-08-16,01:23:16,N/A,N/A,N/A,0.1075,10.5376,-28.5357
A2813,2022-09-15,02:49:34,N/A,N/A,N/A,0.2924,10.8538,-20.6212
A3017,2019-11-20,12:11:41,0.754,0.939,0.651,0.2195,36.4711,-41.9146
A3186,2022-09-10,20:36:16,N/A,N/A,N/A,0.127,57.9406,-73.9725
A3230,2023-10-09,00:21:16,N/A,N/A,N/A,0.1445,62.8683,-63.6858
Abell3322,2021-10-20,15:21:53,N/A,N/A,N/A,0.201,77.5734,-45.3217


In [49]:

# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------

def _is_missing(val) -> bool:
    """Return True if val should be rendered as \\cdots."""
    if val is None:
        return True
    try:
        if np.isnan(val):
            return True
    except (TypeError, ValueError):
        pass
    if isinstance(val, str) and val.strip().upper() in ("N/A", "NAN", ""):
        return True
    return False


def _fmt(val, fmt_spec: str = "") -> str:
    """Format a scalar value, substituting \\cdots for missing data."""
    if _is_missing(val):
        return r"\cdots"
    if fmt_spec:
        try:
            return format(val, fmt_spec)
        except (TypeError, ValueError):
            try:
                return format(float(val), fmt_spec)
            except (TypeError, ValueError):
                return str(val)
    return str(val)


def _coerce_float(val):
    """Try to cast val to float; return np.nan on failure or if already missing."""
    if _is_missing(val):
        return np.nan
    try:
        return float(val)
    except (TypeError, ValueError):
        return np.nan


def _fwhm_cell(avg, fmax, fmin, fmt_spec: str = ".2f") -> str:
    """
    Render FWHM trio as  avg^{+delta_hi}_{-delta_lo}.

    delta_hi = MAX - avg   (how far the max exceeds the mean)
    delta_lo = avg - MIN   (how far the mean exceeds the min)

    If avg is missing the whole cell becomes \\cdots.
    If either bound is missing the exponent / subscript is also \\cdots.
    """
    avg, fmax, fmin = _coerce_float(avg), _coerce_float(fmax), _coerce_float(fmin)

    if _is_missing(avg):
        return r"\cdots"

    avg_s = format(avg, fmt_spec)

    if _is_missing(fmax):
        hi_s = r"\cdots"
    else:
        hi_s = format(fmax - avg, fmt_spec)

    if _is_missing(fmin):
        lo_s = r"\cdots"
    else:
        lo_s = format(avg - fmin, fmt_spec)

    return rf"${avg_s}^{{+{hi_s}}}_{{-{lo_s}}}$"


# ---------------------------------------------------------------------------
# header sanitisation
# ---------------------------------------------------------------------------

_LATEX_SPECIAL = str.maketrans({
    "_": r"\_",
    "^": r"\^{}",
    "&": r"\&",
    "%": r"\%",
    "$": r"\$",
    "#": r"\#",
    "{": r"\{",
    "}": r"\}",
    "~": r"\textasciitilde{}",
})


def _sanitise_header(name: str) -> str:
    """Escape special LaTeX characters in a column name."""
    return name.translate(_LATEX_SPECIAL)


# ---------------------------------------------------------------------------
# main converter
# ---------------------------------------------------------------------------

FWHM_AVG_KEY = "FWHM_XY_AVG"
FWHM_MAX_KEY = "FWHM_MAX"
FWHM_MIN_KEY = "FWHM_MIN"

FWHM_MERGED_HEADER = r"FWHM $\overline{xy}^{+\mathrm{max}}_{-\mathrm{min}}$ [\arcsec]"


def table_to_latex(
    table: Table,
    output_path: str,
    caption: str = "My table.",
    label: str = "tab:mytable",
    col_fmt_specs: dict | None = None,
    fwhm_fmt_spec: str = ".2f",
    default_fmt_spec: str = "",
    fontsize: str = r"\small",
) -> None:
    """
    Write *table* as a LaTeX table to *output_path*.

    Parameters
    ----------
    table          : astropy.table.Table
    output_path    : destination .tex file
    caption        : table caption
    label          : \\label{} value
    col_fmt_specs  : mapping  column_name -> Python format spec  (e.g. ".3f")
                     applied per column; overrides default_fmt_spec
    fwhm_fmt_spec  : format spec used for the merged FWHM column values
    default_fmt_spec: fallback format spec for numeric cells
    fontsize       : LaTeX font-size command inserted before longtable
    """
    col_fmt_specs = col_fmt_specs or {}

    # ------------------------------------------------------------------
    # Work out which columns are present and build output column list
    # ------------------------------------------------------------------
    names = list(table.colnames)
    has_fwhm = all(k in names for k in (FWHM_AVG_KEY, FWHM_MAX_KEY, FWHM_MIN_KEY))

    # Build the ordered list of output columns, merging FWHM trio into one slot
    out_cols = []          # list of column names (or the sentinel "FWHM_MERGED")
    seen_fwhm = False
    for name in names:
        if name in (FWHM_AVG_KEY, FWHM_MAX_KEY, FWHM_MIN_KEY):
            if has_fwhm and not seen_fwhm:
                out_cols.append("FWHM_MERGED")
                seen_fwhm = True
            # skip the other two FWHM columns
        else:
            out_cols.append(name)

    n_cols = len(out_cols)

    # ------------------------------------------------------------------
    # Build header row
    # ------------------------------------------------------------------
    headers = []
    for col in out_cols:
        if col == "FWHM_MERGED":
            headers.append(FWHM_MERGED_HEADER)
        else:
            headers.append(_sanitise_header(col))

    # ------------------------------------------------------------------
    # Build data rows
    # ------------------------------------------------------------------
    rows = []
    for row in table:
        cells = []
        for col in out_cols:
            if col == "FWHM_MERGED":
                cell = _fwhm_cell(
                    row[FWHM_AVG_KEY],
                    row[FWHM_MAX_KEY],
                    row[FWHM_MIN_KEY],
                    fwhm_fmt_spec,
                )
            else:
                val = row[col]
                fmt = col_fmt_specs.get(col, default_fmt_spec)
                cell = _fmt(val, fmt)
            cells.append(cell)
        rows.append(cells)

    # ------------------------------------------------------------------
    # Assemble LaTeX
    # ------------------------------------------------------------------
    col_spec = "l" + "c" * (n_cols - 1)   # first col left-aligned, rest centred

    header_row = "        " + " & ".join(headers) + r" \\"
    continued = f"\\multicolumn{{{n_cols}}}{{r}}{{\\footnotesize (continued)}}"
    cont_head  = f"\\multicolumn{{{n_cols}}}{{l}}{{\\footnotesize (cont.)}}"

    # longtable: no outer {table} float — caption/label live inside the env.
    # \endfirsthead  — header printed only on page 1
    # \endhead       — header repeated on continuation pages
    # \endfoot       — footer on all but the last page
    # \endlastfoot   — footer on the last page only
    lines = [
        r"% Requires \usepackage{booktabs,longtable} in your preamble",
        f"    {fontsize}",
        f"    \\begin{{longtable}}{{{col_spec}}}",
        # caption + label (the \\ after \caption is required by longtable)
        f"    \\caption{{{caption}}} \\label{{{label}}} \\\\",
        # --- first-page header ---
        r"        \toprule",
        header_row,
        r"        \midrule",
        r"    \endfirsthead",
        # --- continuation header ---
        f"        {cont_head} \\\\",
        r"        \toprule",
        header_row,
        r"        \midrule",
        r"    \endhead",
        # --- continuation footer ---
        r"        \midrule",
        f"        {continued} \\\\",
        r"    \endfoot",
        # --- last-page footer ---
        r"        \bottomrule",
        r"    \endlastfoot",
    ]

    for cells in rows:
        lines.append("        " + " & ".join(cells) + r" \\")

    lines += [
        r"    \end{longtable}",
    ]

    latex_str = "\n".join(lines) + "\n"

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(latex_str)

    print(f"LaTeX table written to: {output_path}")



In [50]:
table_to_latex(preprocessed, 'tab.tex')

LaTeX table written to: tab.tex
